# BingePlay Advanced SQL Minor Project

**Database:** `bingeplay`  
**Connection:** MySQL + SQLAlchemy + PyMySQL  
**Dataset:** BingePlay Streaming Platform

> **Important:** Replace the MySQL password in the connection cell before running. The SQL queries below are designed for MySQL 8+ and each question has its own `pandas.read_sql()` execution cell.


In [ ]:
# Run once if the required packages are not installed:
# %pip install pandas sqlalchemy pymysql

import pandas as pd
from sqlalchemy import create_engine, text
from getpass import getpass

DB_USER = "root"
DB_PASSWORD = getpass("Enter MySQL password: ")
DB_HOST = "localhost"
DB_PORT = 3306
DB_NAME = "bingeplay"

engine = create_engine(
    f"mysql+pymysql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

# Test the connection
pd.read_sql("SELECT 1 AS connection_test", engine)


## Q1 — Active revenue

**Answer:** 2,340 active subscriptions  
**Total monthly recurring revenue:** ₹784,260

The active-subscription condition includes subscriptions whose `end_date` is `NULL` as well as subscriptions ending after 30 June 2024.

In [ ]:
query = """
SELECT
    COUNT(*) AS active_subscriptions,
    SUM(monthly_price_inr) AS total_monthly_revenue_inr
FROM subscriptions
WHERE status = 'active'
  AND (end_date IS NULL OR end_date > '2024-06-30');
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q2 — Signup momentum

**Monthly signups**

| Month | Signups |
|---|---:|
| January | 350 |
| February | 400 |
| March | 500 |
| April | 550 |
| May | 600 |
| June | 600 |

**Highest-signup month:** May and June, tied at 600 signups.

In [ ]:
query = """
SELECT
    MONTH(signup_date) AS month_number,
    MONTHNAME(signup_date) AS month,
    COUNT(*) AS signup_count
FROM users
WHERE signup_date >= '2024-01-01'
  AND signup_date < '2024-07-01'
GROUP BY MONTH(signup_date), MONTHNAME(signup_date)
ORDER BY month_number;
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q3 — Device analytics

The NULL `user_id` sessions are excluded as required.

| Device | Sessions | Watch minutes | Avg. minutes | Completion rate |
|---|---:|---:|---:|---:|
| Laptop | 15,105 | 453,434 | 30.02 | 60.51% |
| Mobile | 50,172 | 1,504,355 | 29.98 | 60.24% |
| TV | 27,981 | 840,595 | 30.04 | 59.98% |
| Tablet | 7,091 | 210,733 | 29.72 | 59.79% |

In [ ]:
query = """
SELECT
    device_type,
    COUNT(*) AS total_sessions,
    SUM(watch_minutes) AS total_watch_minutes,
    ROUND(AVG(watch_minutes), 2) AS avg_watch_minutes_per_session,
    ROUND(
        100.0 * SUM(CASE WHEN completed = 1 THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS completion_rate_pct
FROM watch_sessions
WHERE user_id IS NOT NULL
GROUP BY device_type
ORDER BY device_type;
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q4 — Rating distribution

| Stars | Count | Percentage |
|---:|---:|---:|
| 1 | 234 | 4.68% |
| 2 | 352 | 7.04% |
| 3 | 847 | 16.94% |
| 4 | 1,781 | 35.62% |
| 5 | 1,786 | 35.72% |

**Percentage of all ratings that are 4 or 5 stars:** **71.34%**.

In [ ]:
query = """
SELECT
    stars,
    COUNT(*) AS rating_count,
    ROUND(100.0 * COUNT(*) / (SELECT COUNT(*) FROM ratings), 2) AS rating_percentage
FROM ratings
GROUP BY stars
ORDER BY stars;
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))

four_five = pd.read_sql("""
SELECT ROUND(
    100.0 * SUM(CASE WHEN stars IN (4, 5) THEN 1 ELSE 0 END) / COUNT(*),
    2
) AS pct_4_or_5_stars
FROM ratings;
""", engine)
print("\n4-or-5-star percentage:")
print(four_five.to_string(index=False))


## Q5 — Originals vs acquired

| Group | Shows | Avg. IMDb | Avg. release year |
|---|---:|---:|---:|
| Originals | 30 | 7.92 | 2020.37 |
| Acquired | 70 | 6.63 | 2020.73 |

**Interpretation:** Originals perform better on IMDb ratings by **1.29 points** (7.92 − 6.63).

In [ ]:
query = """
SELECT
    CASE WHEN is_original = 1 THEN 'Originals' ELSE 'Acquired' END AS content_group,
    COUNT(*) AS show_count,
    ROUND(AVG(imdb_rating), 2) AS avg_imdb_rating,
    ROUND(AVG(release_year), 2) AS avg_release_year
FROM shows
GROUP BY is_original
ORDER BY is_original DESC;
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q6 — Binge day detection

**Total binge days in Q2 2024:** 414  
**User with the most binge days:** `U02956`  
**That user's binge days:** 8

A binge day is counted at the `(user_id, show_id, session_date)` level with at least 5 sessions.

In [ ]:
query = """
WITH binge_days AS (
    SELECT
        user_id,
        show_id,
        session_date
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date BETWEEN '2024-04-01' AND '2024-06-30'
    GROUP BY user_id, show_id, session_date
    HAVING COUNT(*) >= 5
),
user_binge_counts AS (
    SELECT user_id, COUNT(*) AS binge_days
    FROM binge_days
    GROUP BY user_id
)
SELECT
    (SELECT COUNT(*) FROM binge_days) AS total_binge_days,
    user_id AS top_user_id,
    binge_days AS top_user_binge_days
FROM user_binge_counts
ORDER BY binge_days DESC, user_id
LIMIT 1;
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q7 — Q1 signups who never watched

**Total Q1 2024 signups:** 1,250  
**Q1 signups who never watched:** 226 users

The solution uses `LEFT JOIN ... IS NULL`, which correctly handles the two `NULL` `user_id` values in `watch_sessions`.

In [ ]:
query = """
SELECT
    COUNT(*) AS total_q1_signups,
    SUM(
        CASE
            WHEN ws.user_id IS NULL THEN 1
            ELSE 0
        END
    ) AS never_watched
FROM users u
LEFT JOIN (
    SELECT DISTINCT user_id
    FROM watch_sessions
    WHERE user_id IS NOT NULL
) ws
    ON u.user_id = ws.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-04-01';
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q8 — Over-paying Premium/Family users

**Number of over-paying Premium/Family users:** **6**

These are current Premium/Family users whose watch history contains at least one watched show, but no watched show requires Premium or Family access.

In [ ]:
query = """
WITH current_plan AS (
    SELECT
        user_id,
        plan,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date DESC, subscription_id DESC
        ) AS rn
    FROM subscriptions
    WHERE status = 'active'
      AND start_date <= '2024-06-30'
),
premium_family_users AS (
    SELECT user_id
    FROM current_plan
    WHERE rn = 1
      AND plan IN ('Premium', 'Family')
)
SELECT COUNT(*) AS over_paying_users
FROM premium_family_users p
WHERE EXISTS (
    SELECT 1
    FROM watch_sessions ws
    WHERE ws.user_id = p.user_id
)
AND NOT EXISTS (
    SELECT 1
    FROM watch_sessions ws
    JOIN shows s ON s.show_id = ws.show_id
    WHERE ws.user_id = p.user_id
      AND s.min_plan IN ('Premium', 'Family')
);
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q9 — Upgrade success cohort

**Users in the cohort:** 55  
**Average days from signup to first upgrade:** 64.96 days

The cohort consists of January 2024 signups whose earliest subscription was Basic, who later upgraded to Premium/Family, and who are active as of 30 June 2024.

In [ ]:
query = """
WITH ordered_subscriptions AS (
    SELECT
        s.*,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY start_date, subscription_id
        ) AS rn
    FROM subscriptions s
),
first_basic AS (
    SELECT
        user_id,
        start_date AS first_subscription_date
    FROM ordered_subscriptions
    WHERE rn = 1
      AND plan = 'Basic'
),
first_upgrade AS (
    SELECT
        o.user_id,
        MIN(o.start_date) AS first_upgrade_date
    FROM ordered_subscriptions o
    JOIN first_basic fb
        ON fb.user_id = o.user_id
    WHERE o.start_date > fb.first_subscription_date
      AND o.plan IN ('Premium', 'Family')
    GROUP BY o.user_id
),
active_users AS (
    SELECT DISTINCT user_id
    FROM subscriptions
    WHERE status = 'active'
      AND start_date <= '2024-06-30'
      AND (end_date IS NULL OR end_date > '2024-06-30')
)
SELECT
    COUNT(*) AS cohort_users,
    ROUND(
        AVG(DATEDIFF(fu.first_upgrade_date, u.signup_date)),
        2
    ) AS avg_days_signup_to_first_upgrade
FROM first_upgrade fu
JOIN first_basic fb ON fb.user_id = fu.user_id
JOIN users u ON u.user_id = fu.user_id
JOIN active_users au ON au.user_id = fu.user_id
WHERE u.signup_date >= '2024-01-01'
  AND u.signup_date < '2024-02-01';
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q10 — Cliffhanger comebacks

**Total cliffhanger comeback events:** 4,345  
**Top show:** `S088` — **Rayalaseema Raga**  
**Comeback events for the top show:** 64

Each event is counted once per `(user_id, show_id, incomplete_date)`.

In [ ]:
query = """
WITH comeback_events AS (
    SELECT DISTINCT
        w1.user_id,
        w1.show_id,
        w1.session_date AS incomplete_date
    FROM watch_sessions w1
    JOIN watch_sessions w2
        ON w2.user_id = w1.user_id
       AND w2.show_id = w1.show_id
       AND w2.session_date > w1.session_date
       AND w2.session_date <= DATE_ADD(w1.session_date, INTERVAL 7 DAY)
    WHERE w1.completed = 0
      AND w1.user_id IS NOT NULL
),
show_counts AS (
    SELECT
        show_id,
        COUNT(*) AS comeback_events
    FROM comeback_events
    GROUP BY show_id
)
SELECT
    (SELECT COUNT(*) FROM comeback_events) AS total_comeback_events,
    sc.show_id AS top_show_id,
    s.title AS top_show_title,
    sc.comeback_events AS top_show_events
FROM show_counts sc
JOIN shows s ON s.show_id = sc.show_id
ORDER BY sc.comeback_events DESC, sc.show_id
LIMIT 1;
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q11 — Consecutive-week engagement

**Users with a streak of 4+ consecutive weeks:** 1,675  
**Longest streak:** 26 weeks  
**One user with the longest streak:** `U00213`

The solution uses the gaps-and-islands pattern on distinct ISO calendar weeks.

In [ ]:
query = """
WITH distinct_weeks AS (
    SELECT DISTINCT
        user_id,
        YEARWEEK(session_date, 3) AS iso_week
    FROM watch_sessions
    WHERE user_id IS NOT NULL
),
numbered_weeks AS (
    SELECT
        user_id,
        iso_week,
        ROW_NUMBER() OVER (
            PARTITION BY user_id
            ORDER BY iso_week
        ) AS rn
    FROM distinct_weeks
),
islands AS (
    SELECT
        user_id,
        iso_week,
        iso_week - rn AS island_key
    FROM numbered_weeks
),
streaks AS (
    SELECT
        user_id,
        island_key,
        COUNT(*) AS streak_weeks
    FROM islands
    GROUP BY user_id, island_key
)
SELECT
    (SELECT COUNT(DISTINCT user_id)
     FROM streaks
     WHERE streak_weeks >= 4) AS users_with_4plus_streak,
    s.streak_weeks AS longest_streak_weeks,
    s.user_id AS one_user_with_longest_streak
FROM streaks s
ORDER BY s.streak_weeks DESC, s.user_id
LIMIT 1;
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))


## Q12 — Churn signal detection

**Total churn-signal users:** **521**

A churn signal is a user with some May 2024 watch activity whose June watch minutes fell by at least 50% compared with May. The code cell prints the complete user-level result requested by the brief.

In [ ]:
query = """
WITH monthly_watch AS (
    SELECT
        user_id,
        SUM(
            CASE
                WHEN session_date >= '2024-05-01'
                 AND session_date < '2024-06-01'
                THEN watch_minutes ELSE 0
            END
        ) AS may_watch_minutes,
        SUM(
            CASE
                WHEN session_date >= '2024-06-01'
                 AND session_date < '2024-07-01'
                THEN watch_minutes ELSE 0
            END
        ) AS june_watch_minutes
    FROM watch_sessions
    WHERE user_id IS NOT NULL
      AND session_date >= '2024-05-01'
      AND session_date < '2024-07-01'
    GROUP BY user_id
),
churn_signals AS (
    SELECT
        user_id,
        may_watch_minutes,
        june_watch_minutes,
        ROUND(
            100.0 * (may_watch_minutes - june_watch_minutes)
            / may_watch_minutes,
            2
        ) AS drop_percentage
    FROM monthly_watch
    WHERE may_watch_minutes > 0
      AND june_watch_minutes <= may_watch_minutes * 0.5
)
SELECT
    cs.user_id,
    u.name,
    cs.may_watch_minutes,
    cs.june_watch_minutes,
    cs.drop_percentage
FROM churn_signals cs
JOIN users u ON u.user_id = cs.user_id
ORDER BY cs.user_id;
"""

result = pd.read_sql(query, engine)
print(result.to_string(index=False))
print("\nTotal churn-signal users:", len(result))


## Final verification

Before submission, run all cells from top to bottom and confirm that the outputs from your MySQL database match the numeric answers written above. The project brief specifically requires the notebook to contain one code cell per question, use SQLAlchemy + PyMySQL, execute queries with `pandas.read_sql`, print outputs, and include a markdown answer for each question.

Also make sure the database row counts are:

- `users`: 3000
- `subscriptions`: 4497
- `shows`: 100
- `watch_sessions`: 100351
- `ratings`: 5000
